In [ ]:
# Credit Scoring and Expected Loss using LightGBM

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import matplotlib.pyplot as plt

# Load dataset
data = pd.read_csv("./dataset/application_train.csv")

# Select features
selected_features = [
    "CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS", "CNT_CHILDREN",
    "CNT_FAM_MEMBERS", "NAME_EDUCATION_TYPE", "OCCUPATION_TYPE",
    "NAME_INCOME_TYPE", "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH",
    "NAME_HOUSING_TYPE", "REGION_POPULATION_RELATIVE", "DAYS_LAST_PHONE_CHANGE",
    "TARGET"
]

df = data[selected_features].copy()
df = pd.get_dummies(df, drop_first=True)
X = df.drop(columns="TARGET")
y = df["TARGET"]

# Impute missing
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train LightGBM model
lgb_model = lgb.LGBMClassifier()
lgb_model.fit(X_train, y_train)

lgb_pred_prob = lgb_model.predict_proba(X_test)[:, 1]

# Compute AUC
auc = roc_auc_score(y_test, lgb_pred_prob)
print(f"LightGBM AUC: {auc:.4f}")

# Convert PD to Credit Score
pd_prob = np.clip(lgb_pred_prob, 1e-6, 1 - 1e-6)
log_odds = np.log(pd_prob / (1 - pd_prob))
PDO = 20
factor = PDO / np.log(2)
offset = 487.7
credit_scores = offset - factor * log_odds
credit_scores = np.clip(credit_scores, 300, 850)

# Expected Loss = PD * LGD * EAD
LGD = 0.5
EAD = data.loc[y_test.index, "AMT_CREDIT"].values
expected_loss = pd_prob * LGD * EAD
total_EL = expected_loss.sum()

# Print total EL
print(f"Total Expected Loss on test set: {total_EL:,.2f}")

# Show sample results
summary_df = pd.DataFrame({
    "PD": pd_prob,
    "CreditScore": credit_scores,
    "AMT_CREDIT": EAD,
    "ExpectedLoss": expected_loss
})
summary_df.head(20)


[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2456
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 50
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606
LightGBM AUC: 0.6895
Total Expected Loss on test set: 1,379,237,517.61


/opt/anaconda3/envs/ml-anaconda-python12-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,PD,CreditScore,AMT_CREDIT,ExpectedLoss
0,0.108245,548.546929,465457.5,25191.732348
1,0.066775,563.796878,1281712.5,42793.205558
2,0.055104,569.698592,495000.0,13638.213287
3,0.022242,596.862586,254700.0,2832.498984
4,0.200750,527.564891,308133.0,30928.889075
5,0.059057,567.578363,152820.0,4512.572432
6,0.065243,564.513929,900000.0,29359.347922
7,0.136188,541.002477,810000.0,55155.938888
8,0.060320,566.929312,781920.0,23582.640106
9,0.109807,548.083064,900000.0,49412.995711


In [3]:
def convert_pd_to_score_and_el(pd_array, ead_array, lgd=0.5, pdo=20, pd_ref=0.02, score_ref=600, score_range=(300, 850)):
    """
    Chuyển đổi PD (Probability of Default) thành Credit Score và tính Expected Loss (EL).
    
    Parameters:
    - pd_array: ndarray hoặc list các giá trị PD (xác suất vỡ nợ)
    - ead_array: ndarray hoặc list các giá trị EAD (dư nợ khoản vay)
    - lgd: Loss Given Default (tỉ lệ tổn thất khi vỡ nợ), mặc định 50%
    - pdo: Points to Double Odds (mặc định = 20)
    - pd_ref: PD tham chiếu, ví dụ 0.02 để tính offset
    - score_ref: điểm tín dụng tương ứng với PD tham chiếu (ví dụ 600)
    - score_range: giới hạn điểm tín dụng (300–850)

    Returns:
    - DataFrame chứa các cột: PD, CreditScore, EAD, ExpectedLoss
    """
    pd_array = np.clip(np.array(pd_array), 1e-6, 1 - 1e-6)
    ead_array = np.array(ead_array)

    # Tính hệ số và offset
    factor = pdo / np.log(2)
    offset = score_ref + factor * np.log(pd_ref / (1 - pd_ref))

    # Tính log-odds và điểm tín dụng
    log_odds = np.log(pd_array / (1 - pd_array))
    credit_scores = offset - factor * log_odds
    credit_scores = np.clip(credit_scores, score_range[0], score_range[1])

    # Tính Expected Loss
    expected_loss = pd_array * lgd * ead_array

    return pd.DataFrame({
        "PD": pd_array,
        "CreditScore": credit_scores,
        "EAD": ead_array,
        "ExpectedLoss": expected_loss
    })

In [2]:
# new_customer là vector chứa thông tin đã chuẩn hóa của 1 khách hàng (giống cấu trúc X_test[0])
new_customer = X_test[0].reshape(1, -1)

# Dự đoán xác suất nợ xấu (PD)
pd_single = lgb_model.predict_proba(new_customer)[0, 1]

print(f"Xác suất nợ xấu của khách hàng mới: {pd_single:.4f}")

Xác suất nợ xấu của khách hàng mới: 0.1082


/opt/anaconda3/envs/ml-anaconda-python12-env/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [4]:
# Ví dụ khách hàng mới:
pd_array = [pd_single]
ead_array = [20000000]  # khoản vay 20 triệu

result = convert_pd_to_score_and_el(pd_array, ead_array)
print(result)

         PD  CreditScore       EAD  ExpectedLoss
0  0.108245   548.552732  20000000  1.082450e+06
